<img src="https://devra.ai/analyst/notebook/3839/image.jpg" style="width: 100%; height: auto;" />

<div style="text-align:center; border-radius:15px; padding:15px; color:white; margin:0; font-family: 'Orbitron', sans-serif; background: #2E0249; background: #11001C; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.3); overflow:hidden; margin-bottom: 1em;">
  <div style="font-size:150%; color:#FEE100"><b>NanoFluid Thermal Conductivity Analysis</b></div>
  <div>This notebook was created with the help of <a href="https://devra.ai/ref/kaggle" style="color:#6666FF">Devra AI</a></div>
</div>

## Introduction

An interesting observation about nanofluid thermal conductivity measurements is that the interplay between particle material and base fluid properties can yield counterintuitive trends in the experimental values. This dataset provides an opportunity to explore such interactions by delving into the thermophysical parameters and uncovering hidden patterns. If you find this notebook useful, please consider upvoting it.

## Table of Contents

- [Data Import and Setup](#Data-Import-and-Setup)
- [Data Exploration](#Data-Exploration)
- [Data Cleaning and Preprocessing](#Data-Cleaning-and-Preprocessing)
- [Exploratory Data Analysis](#Exploratory-Data-Analysis)
- [Predictor Modeling](#Predictor-Modeling)
- [Summary and Future Directions](#Summary-and-Future-Directions)

In [ ]:
# Imports and initial setup
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')  # Use Agg backend for Matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns

# Set seaborn style for better aesthetics
sns.set(style='whitegrid')

# For reproducibility
np.random.seed(42)

## Data Import and Setup

Let's import the dataset. Note that this dataset uses a comma as the delimiter and UTF-8-SIG encoding. If you encounter any encoding issues, make sure to specify the correct encoding in pandas.read_csv.

In [ ]:
# Load the dataset
file_path = '/kaggle/input/nanofluid-thermal-conductivity-prediction/Dataset_Thermal_Conductivity.csv'
df = pd.read_csv(file_path, delimiter=',', encoding='UTF-8-SIG')

# Display the first few rows
print('Dataset head:')
print(df.head())

## Data Exploration

Let's get a sense of the dataset dimensions, data types, and statistical summaries. We will look at the shape of the dataset and check for any missing values that might need addressing later.

In [ ]:
# Basic dataset information
print('Dataset shape: ', df.shape)
print('\nDataset info:')
print(df.info())

print('\nStatistical Summary:')
print(df.describe(include='all'))

# Check for missing values
print('\nMissing Values:')
print(df.isnull().sum())

## Data Cleaning and Preprocessing

We observe the following columns:
- Particle Material: string
- Base Fluid: string
- Temperature (°C): integer
- Particle Size (nm): integer
- Particle Volume Fraction (in %): number
- Thermal Conductivity of Liquid (km): number
- Thermal Conductivity of Particle (kp): integer
- Exp-TC: number
- KKL-TC: number

No explicit date columns are present, so no date parsing is needed. However, always ensure that numeric columns are in the correct format and check for outliers or missing values.

In [ ]:
# Data Cleaning

# Rename columns for easier usage if necessary (e.g., remove spaces and special characters)
df.columns = [col.strip().replace(' ', '_').replace('(', '').replace(')', '').replace('\xB0C', 'C') for col in df.columns]

# Check for missing values again
missing_values = df.isnull().sum()
print('Missing values after renaming columns:')
print(missing_values)

# If missing values were present, one might consider imputing or dropping them depending on context.
# For example, to drop missing values:
# df.dropna(inplace=True)

# Convert appropriate columns to numeric if necessary
numeric_columns = ['Temperature_C', 'Particle_Size_nm', 'Particle_Volume_Fraction_in_%', 
                   'Thermal_Conductivity_of_Liquid_km', 'Thermal_Conductivity_of_Particle_kp',
                   'Exp-TC', 'KKL-TC']

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# After conversion, check again for any conversion issues
print('\nDataset info after numeric conversion:')
print(df.info())

## Exploratory Data Analysis

We now visualize several aspects of the dataset. We'll use multiple visualization methods for a thorough investigation. First, let's look at the distribution of key numeric features using histograms and box plots. Next, we will visualize pairwise patterns with a pair plot. Finally, with four or more numeric columns present, we construct a correlation heatmap.

In [ ]:
# Extract numeric data for correlation analysis
numeric_df = df.select_dtypes(include=[np.number])

print('Numeric columns available for analysis:', numeric_df.columns.tolist())

# Histogram for numeric features
numeric_columns = numeric_df.columns
plt.figure(figsize=(15, 10))
for idx, col in enumerate(numeric_columns):
    plt.subplot(3, 3, idx+1)
    sns.histplot(numeric_df[col].dropna(), kde=True, color='skyblue')
    plt.title(f'Histogram of {col}')
plt.tight_layout()
plt.show()

# Boxplots to identify outliers
plt.figure(figsize=(15, 10))
for idx, col in enumerate(numeric_columns):
    plt.subplot(3, 3, idx+1)
    sns.boxplot(x=numeric_df[col].dropna(), color='lightgreen')
    plt.title(f'Boxplot of {col}')
plt.tight_layout()
plt.show()

# Pair Plot for exploring pairwise relationships
sns.pairplot(numeric_df.dropna())
plt.show()

# Correlation Heatmap if there are four or more numeric columns
if numeric_df.shape[1] >= 4:
    plt.figure(figsize=(10, 8))
    correlation_matrix = numeric_df.corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', square=True, fmt='.2f')
    plt.title('Correlation Heatmap')
    plt.show()
else:
    print('Not enough numeric columns for correlation heatmap.')

## Predictor Modeling

Based on the analysis, it appears that predicting the experimental thermal conductivity (Exp-TC) may be useful. We will build a simple regression model using selected features. In this example, we use a Linear Regression model from scikit-learn. The predictor's performance is evaluated using metrics such as the R2 score and Mean Squared Error.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Select a subset of features for prediction. Here we drop categorical features.
features = ['Temperature_C', 'Particle_Size_nm', 'Particle_Volume_Fraction_in_%', 'Thermal_Conductivity_of_Liquid_km', 'Thermal_Conductivity_of_Particle_kp']
target = 'Exp-TC'

# Drop any rows with missing values in the selected columns
model_df = df[features + [target]].dropna()

X = model_df[features]
y = model_df[target]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model performance
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print('Linear Regression Model Evaluation:')
print(f'Mean Squared Error: {mse}')
print(f'R^2 Score: {r2}')

## Summary and Future Directions

This notebook provided a comprehensive analysis of the nanofluid thermal conductivity dataset. We began by exploring the dataset and performing necessary cleaning and preprocessing. Next, multiple visualizations were applied, ranging from histograms to a correlation heatmap, to identify patterns and potential outliers. Finally, a linear regression model was built to predict the experimental thermal conductivity (Exp-TC) along with its performance evaluation.

In future analyses, one could consider:

- Experimenting with additional features or non-linear models to capture more complex relationships.
- Applying cross-validation to better assess model robustness.
- Investigating the impact of particle material and base fluid using advanced encoding techniques for categorical data.
- Tuning hyperparameters with grid search or randomized search to improve prediction accuracy.

Thank you for exploring this notebook. If you found it useful, please consider upvoting.